# Adversarial Search: Solving Tic-Tac-Toe with Monte Carlo Tree Search

## Introduction 

Multiplayer games can be implemented as:
1. Nondeterministic actions: The opponent is seen as part of an environment with nondeterministic actions. Non-determinism is the result of the unknown opponent's moves. 
2. Optimal Decisions: Minimax search (search complete game tree) and alpha-beta pruning.
3. Heuristic Alpha-Beta Tree Search: Cut off tree search and use heuristic to estimate state value. 
4. __Monte Carlo Search:__ Simulate playouts to estimate state value. 

Here we will implement search for Tic-Tac-Toe (see [rules](https://en.wikipedia.org/wiki/Tic-tac-toe)). The game is a __zero-sum game__: Win by x results in +1, win by o in -1 and a tie has a value of 0. Max plays x and tries to maximize the outcome while Min plays o and tries to minimize the outcome.   

We will implement
* We enhance Pure Monte Carlo Search by using the upper confidence bound (UCB1) selection policy. That is, we use UCB1 to determine for which action to perform the next playout. This will allow the algorithm to focus on actions here it needs to collect more information. Note that complete Upper Confidence Bounds applied to Trees (UCT) creates a tree and the expand step in the code needs to be added. The tree branch for the chosen move is preserved and used for future moves.

Môi trường Tic-Tac-Toe và các hàm trạng thái được cài đặt trực tiếp trong notebook để notebook có thể chạy độc lập, không phụ thuộc file `tictactoe.py`.

In [36]:
import os, math, time, random
import numpy as np, pandas as pd
from collections import Counter
np.random.seed(2026); random.seed(2026)

import matplotlib.pyplot as plt
import os

# Tạo thư mục result tại thư mục đang chạy notebook
RESULT_DIR = "result"
os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Thư mục lưu kết quả: {os.path.abspath(RESULT_DIR)}")
_WIN_LINES = (
    (0,1,2), (3,4,5), (6,7,8),
    (0,3,6), (1,4,7), (2,5,8),
    (0,4,8), (2,4,6)
)

def empty_board():
    return [' '] * 9

def actions(board):
    return [i for i, value in enumerate(board) if value == ' ']

def result(state, player, action):
    if action not in actions(state):
        raise ValueError(f"Illegal move {action}")
    new_state = list(state)
    new_state[action] = player
    return new_state

def other(player):
    return 'o' if player == 'x' else 'x'

def check_board(state):
    for a, b, c in _WIN_LINES:
        if state[a] != ' ' and state[a] == state[b] == state[c]:
            return state[a]
    return 'd' if ' ' not in state else 'n'

def terminal(state):
    return check_board(state) != 'n'

def utility(state, player='x'):
    outcome = check_board(state)
    if outcome == player:
        return 1
    if outcome == other(player):
        return -1
    if outcome == 'd':
        return 0
    return None

def show_board(board):
    print(np.asarray(board).reshape(3, 3))

def random_player(board, player='x'):
    return random.choice(actions(board))

def play(player1, player2, N=100):
    counts = {'x': 0, 'o': 0, 'd': 0}
    for game in range(N):
        board = empty_board()
        current = 'x'
        first, second = (player1, player2) if game % 2 == 0 else (player2, player1)

        while not terminal(board):
            agent = first if current == 'x' else second
            move = agent(board, current)
            board = result(board, current, move)
            current = other(current)

        counts[check_board(board)] += 1

    return counts


Thư mục lưu kết quả: f:\TTNTNC\LAB05_GAME\result


In [37]:
# The complete Tic-Tac-Toe environment is already defined in the preceding cell.
# Do not import tictactoe.py: this notebook is self-contained.
assert len(empty_board()) == 9
assert len(actions(empty_board())) == 9
print("Using the self-contained Tic-Tac-Toe environment.")


Using the self-contained Tic-Tac-Toe environment.


# Monte Carlo Search with Upper Confidence Bound

See AIMA page 163. 

We enhance pure Monte Carlo Search by using UCB1 as a __selection policy__. This can be seen as a 
restricted version of UCT: 
* Only build a tree of depth 1 and use the UCB1 selection policy.
* Use a random playout policy.


__Note on the playout policy:__ We use here a random playout policy, which ends up creating just a randomized search that works fine for this toy problem. For real applications you need to extend the code with a good __playout policy__ (e.g., manually created heuristics or a neural [network learned by self-play using reinforcement learning](https://towardsdatascience.com/how-to-teach-an-ai-to-play-games-deep-reinforcement-learning-28f9b920440a)).

## Simulate playouts

In [38]:
def playout(state, action, player='x'):
    """Random rollout after taking action; reward is from `player`'s perspective."""
    state = result(state, player, action)
    current_player = other(player)

    while True:
        outcome = utility(state, player)
        if outcome is not None:
            return outcome

        legal = actions(state)
        chosen = random.choice(legal)
        state = result(state, current_player, chosen)
        current_player = other(current_player)


## Upper Confidence Bound Applied to Trees (Restricted to Depth 1)

In [39]:
import pandas as pd
import math

DEBUG = 1

def UCT_depth1(board, N=100, player='x'):
    """
    Depth-1 UCT using UCB1.

    UCB1(a) = Q(a)/n(a) + C * sqrt(log(N_parent)/n(a))
    where Q(a) is accumulated reward and n(a) is the number of
    simulations assigned to action a. Every legal action is tried
    once before the algorithm uses the UCB1 score, provided N is at least
    the number of legal actions.
    """
    global DEBUG

    if not isinstance(N, (int, np.integer)) or N <= 0:
        raise ValueError("N must be a positive integer.")

    acts = actions(board)
    if not acts:
        return None

    C = math.sqrt(2)
    total_reward = [0.0] * len(acts)
    visits = [0] * len(acts)
    parent_visits = 0

    for _ in range(N):
        # Exploration guarantee: visit every action once when budget allows.
        unvisited = [i for i, count in enumerate(visits) if count == 0]

        if unvisited:
            action_id = unvisited[0]
        else:
            ucb_scores = [
                total_reward[i] / visits[i]
                + C * math.sqrt(math.log(parent_visits) / visits[i])
                for i in range(len(acts))
            ]
            action_id = max(range(len(acts)), key=lambda i: ucb_scores[i])

        reward = playout(board, acts[action_id], player=player)

        total_reward[action_id] += reward
        visits[action_id] += 1
        parent_visits += 1

    # Root action selection: choose the most visited action; break ties by mean reward.
    best_id = max(
        range(len(acts)),
        key=lambda i: (
            visits[i],
            total_reward[i] / visits[i] if visits[i] else -math.inf
        )
    )

    if DEBUG >= 1:
        final_ucb = [
            (total_reward[i] / visits[i]
             + C * math.sqrt(math.log(parent_visits) / visits[i]))
            if visits[i] else math.inf
            for i in range(len(acts))
        ]
        display(pd.DataFrame({
            "action": acts,
            "total utility": total_reward,
            "# of playouts": visits,
            "mean reward": [
                total_reward[i] / visits[i] if visits[i] else np.nan
                for i in range(len(acts))
            ],
            "UCB1 (final)": final_ucb
        }))
        print(f"Best action: {acts[best_id]}")

    return acts[best_id]


In [40]:
board = empty_board()
display(board)

%timeit -n 1 -r 1 UCT_depth1(board, N = 1000)

[' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ']

,action,total utility,# of playouts,mean reward,UCB1 (final)
0,0,25.0,86,0.290698,0.691504
1,1,9.0,50,0.180000,0.705652
2,2,37.0,107,0.345794,0.705123
3,3,42.0,115,0.365217,0.711822
4,4,159.0,320,0.496875,0.704657
5,5,-5.0,14,-0.357143,0.636246
6,6,57.0,144,0.395833,0.705577
7,7,-5.0,14,-0.357143,0.636246
8,8,60.0,150,0.400000,0.703485


Best action: 4
38.8 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## Some Tests

### x is about to win (play 8)

In [41]:
board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[3] = 'o'
board[4] = 'x'

print("Board:")
show_board(board)

print()
%timeit -n1 -r1 UCT_depth1(board)

Board:
[['x' 'o' ' ']
 ['o' 'x' ' ']
 [' ' ' ' ' ']]



,action,total utility,# of playouts,mean reward,UCB1 (final)
0,2,16.0,20,0.800000,1.478614
1,5,12.0,16,0.750000,1.508714
2,6,16.0,19,0.842105,1.538349
3,7,8.0,13,0.615385,1.457102
4,8,32.0,32,1.000000,1.536492


Best action: 8
6.07 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### o is about to win

In [42]:
board = empty_board() 
board[0] = 'o'
board[1] = 'o'
board[3] = 'o'
board[4] = 'x'
board[8] = 'x'

print("Board:")
show_board(board)

print()
%timeit -n1 -r1 UCT_depth1(board, N = 1000)

Board:
[['o' 'o' ' ']
 ['o' 'x' ' ']
 [' ' ' ' 'x']]



,action,total utility,# of playouts,mean reward,UCB1 (final)
0,2,-9.0,433,-0.020785,0.157839
1,5,-12.0,14,-0.857143,0.136246
2,6,-4.0,530,-0.007547,0.153906
3,7,-15.0,23,-0.652174,0.122858


Best action: 6
15.5 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### x can draw if it chooses 7

In [43]:
board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[2] = 'x'
board[4] = 'o'

print("Board:")
show_board(board)

print()
%timeit -n1 -r1 UCT_depth1(board, N = 1000)

Board:
[['x' 'o' 'x']
 [' ' 'o' ' ']
 [' ' ' ' ' ']]



,action,total utility,# of playouts,mean reward,UCB1 (final)
0,3,-12.0,35,-0.342857,0.285417
1,5,-11.0,47,-0.234043,0.308126
2,6,-11.0,57,-0.192982,0.299336
3,7,155.0,839,0.184744,0.313066
4,8,-11.0,22,-0.500000,0.292450


Best action: 7
20.5 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Empty board: Only a draw an be guaranteed

In [44]:
board = empty_board() 

print("Board:")
show_board(board)


print()
%timeit -n1 -r1 UCT_depth1(board, N = 100)
%timeit -n1 -r1 UCT_depth1(board, N = 5000)

Board:
[[' ' ' ' ' ']
 [' ' ' ' ' ']
 [' ' ' ' ' ']]



,action,total utility,# of playouts,mean reward,UCB1 (final)
0,0,3.0,12,0.250000,1.126087
1,1,-2.0,2,-1.000000,1.145966
2,2,2.0,10,0.200000,1.159705
3,3,7.0,18,0.388889,1.104211
4,4,1.0,10,0.100000,1.059705
5,5,12.0,23,0.521739,1.154550
6,6,-1.0,5,-0.200000,1.157228
7,7,-1.0,5,-0.200000,1.157228
8,8,6.0,15,0.400000,1.183596


Best action: 5
8.39 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


,action,total utility,# of playouts,mean reward,UCB1 (final)
0,0,180.0,486,0.370370,0.557587
1,1,13.0,98,0.132653,0.549571
2,2,136.0,391,0.347826,0.556551
3,3,14.0,102,0.137255,0.545916
4,4,1224.0,2520,0.485714,0.567932
5,5,-4.0,39,-0.102564,0.558329
6,6,247.0,631,0.391442,0.555746
7,7,28.0,138,0.202899,0.554235
8,8,231.0,595,0.388235,0.557437


Best action: 4
117 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### A bad situation

In [45]:
board = empty_board() 
board[0] = 'o'
board[2] = 'x'
board[8] = 'o'

print("Board:")
show_board(board)


print()
display(UCT_depth1(board))

Board:
[['o' ' ' 'x']
 [' ' ' ' ' ']
 [' ' ' ' 'o']]



,action,total utility,# of playouts,mean reward,UCB1 (final)
0,1,-4.0,6,-0.666667,0.572307
1,3,-4.0,4,-1.000000,0.517427
2,4,23.0,76,0.302632,0.650753
3,5,-4.0,4,-1.000000,0.517427
4,6,-4.0,6,-0.666667,0.572307
5,7,-4.0,4,-1.000000,0.517427


Best action: 4


4

__Note:__ It looks like random player o is very unlikely to block x and take advantage of the trap by playing the bottom left corner!

## Experiments

### Pure Monte Carlo Search vs. Random

In [46]:
def ucb1_10_player(board, player = 'x'):
    action = UCT_depth1(board, N = 10, player = player)
    return action

def ucb1_100_player(board, player = 'x'):
    action = UCT_depth1(board, N = 100, player = player)
    return action

In [47]:

# Tournament UCB1 N=10 vs Random, 100 games for each starting position.
def tournament_ucb(agent,n=100,agent_first=True,seed=2026):
    counts=Counter(); t0=time.perf_counter()
    for i in range(n):
        np.random.seed(seed+i)
        random.seed(seed+i)
        board=empty_board(); current='x'
        while not terminal(board):
            if (current=='x')==agent_first:
                move=agent(board,current)
            else:
                move=random_player(board,current)
            board=result(board,current,move); current=other(current)
        counts[check_board(board)]+=1
    return {"Agent":agent.__name__,"Vị trí":"X (đi trước)" if agent_first else "O (đi sau)",
            "Agent thắng":counts['x'] if agent_first else counts['o'],
            "Random thắng":counts['o'] if agent_first else counts['x'],
            "Hòa":counts['d'],"Số ván":n,"Thời gian (s)":round(time.perf_counter()-t0,3)}

DEBUG=0
match10=pd.DataFrame([tournament_ucb(ucb1_10_player,100,True,101),
                      tournament_ucb(ucb1_10_player,100,False,201)])
display(match10)


,Agent,Vị trí,Agent thắng,Random thắng,Hòa,Số ván,Thời gian (s)
0,ucb1_10_player,X (đi trước),92,6,2,100,0.057
1,ucb1_10_player,O (đi sau),74,18,8,100,0.046


In [48]:

# Tournament UCB1 N=100 vs Random, 100 games for each starting position.
match100=pd.DataFrame([tournament_ucb(ucb1_100_player,100,True,301),
                       tournament_ucb(ucb1_100_player,100,False,401)])
display(match100)


,Agent,Vị trí,Agent thắng,Random thắng,Hòa,Số ván,Thời gian (s)
0,ucb1_100_player,X (đi trước),98,0,2,100,0.519
1,ucb1_100_player,O (đi sau),87,7,6,100,0.392


In [49]:

# So sánh PMCS (chia đều rollout) và UCB1 (phân bổ rollout thích nghi).
def pmcs_player_10(board,player='x'):
    acts=actions(board)
    if not acts: return None
    vals=[]
    base,rem=divmod(10,len(acts))
    for i,a in enumerate(acts):
        sims=base+(1 if i<rem else 0)
        vals.append(np.mean([playout(board,a,player) for _ in range(sims)]))
    return acts[int(np.argmax(vals))]

def pmcs_player_100(board,player='x'):
    acts=actions(board)
    if not acts: return None
    base=100//len(acts); rem=100%len(acts)
    vals=[]
    for i,a in enumerate(acts):
        n=base+(1 if i<rem else 0)
        vals.append(np.mean([playout(board,a,player) for _ in range(n)]))
    return acts[int(np.argmax(vals))]

comparison_rows=[]
for n,pmcs,ucb in [(10,pmcs_player_10,ucb1_10_player),(100,pmcs_player_100,ucb1_100_player)]:
    for label,agent in [("PMCS",pmcs),("UCB1",ucb)]:
        for first in [True,False]:
            d=tournament_ucb(agent,100,first,seed=7000+n+(0 if first else 100))
            comparison_rows.append({"Phương pháp":label,"Ngân sách":n,"Vị trí":d["Vị trí"],
                                    "Agent thắng":d["Agent thắng"],"Random thắng":d["Random thắng"],
                                    "Hòa":d["Hòa"],"Số ván":d["Số ván"]})
comparison_df=pd.DataFrame(comparison_rows)
display(comparison_df)


,Phương pháp,Ngân sách,Vị trí,Agent thắng,Random thắng,Hòa,Số ván
0,PMCS,10,X (đi trước),94,5,1,100
1,PMCS,10,O (đi sau),74,18,8,100
2,UCB1,10,X (đi trước),89,5,6,100
3,UCB1,10,O (đi sau),74,18,8,100
4,PMCS,100,X (đi trước),97,1,2,100
5,PMCS,100,O (đi sau),87,9,4,100
6,UCB1,100,X (đi trước),98,0,2,100
7,UCB1,100,O (đi sau),87,8,5,100


In [50]:

# Độ ổn định: lặp lại cùng trạng thái với 30 seed độc lập.
stability_board=['x','o','x',' ','o',' ',' ',' ',' ']
stability_rows=[]
for n in [10,100,1000]:
    selected=[]
    for seed in range(30):
        np.random.seed(90000+seed)
        random.seed(90000+seed)
        selected.append(UCT_depth1(stability_board,N=n,player='x'))
    counts=Counter(selected)
    stability_rows.append({"N":n,"Số seed":30,"Nước đi phổ biến nhất":counts.most_common(1)[0][0],
                           "Tỷ lệ phổ biến nhất (%)":round(100*counts.most_common(1)[0][1]/30,1),
                           "Số nước đi khác nhau":len(counts)})
stability_df=pd.DataFrame(stability_rows)
display(stability_df)


,N,Số seed,Nước đi phổ biến nhất,Tỷ lệ phổ biến nhất (%),Số nước đi khác nhau
0,10,30,3,43.3,4
1,100,30,7,96.7,2
2,1000,30,7,100.0,1


## Phân tích và trả lời câu hỏi

**Exploration và exploitation:** UCB1 cộng phần thưởng trung bình với bonus khám phá. Khi một hành động ít được thử, bonus lớn khiến nó tiếp tục được lấy mẫu; khi đã có nhiều mẫu, trung bình phần thưởng chi phối quyết định. Vì vậy UCB1 phân bổ mô phỏng thích nghi thay vì chia đều như PMCS.

**Ảnh hưởng ngân sách mô phỏng:** N nhỏ có phương sai cao và kết quả phụ thuộc seed; tăng N thường làm ước lượng ổn định hơn nhưng tăng thời gian. Bảng `stability_df` định lượng tỷ lệ chọn nước đi phổ biến nhất qua 30 seed.

**Giới hạn:** Đây là UCT giới hạn độ sâu 1; không lưu cây con và rollout vẫn ngẫu nhiên. Do đó nó không tương đương MCTS đầy đủ.

### Phạm vi so sánh PMCS và UCB1

Notebook cài đặt PMCS với ngân sách rollout bằng nhau cho từng nước đi và so sánh trực tiếp với UCB1 trên cùng môi trường, cùng ngân sách mô phỏng (10 và 100), cùng số ván và hai vị trí X/O. Đây là phép so sánh nội bộ được tái lập trong notebook; nếu yêu cầu chấm điểm đòi hỏi đối chiếu đúng các con số đã báo cáo ở Bài 5/TV3, cần bổ sung bảng kết quả gốc của Bài 5 để tránh xem hai lần chạy khác nhau là cùng một thí nghiệm.


In [51]:
from IPython.display import Markdown, display



# Test the same five categories of positions used in the notebook.
uct_test_states = {
    "X sắp thắng": ['x','o',' ','o','x',' ',' ',' ',' '],
    "O sắp thắng": ['o','o',' ','o','x',' ',' ',' ','x'],
    "X có thể hòa": ['x','o','x',' ','o',' ',' ',' ',' '],
    "Bàn trống": empty_board(),
    "Tình huống bất lợi": ['o',' ','x',' ',' ',' ',' ',' ','o']
}

uct_test_rows = []
DEBUG = 0
for state_name, board in uct_test_states.items():
    for budget in [10, 100]:
        selected = UCT_depth1(board, N=budget, player='x')
        assert selected in actions(board), f"Nước đi không hợp lệ: {state_name}"
        uct_test_rows.append({
            "Trạng thái": state_name,
            "Ngân sách mô phỏng": budget,
            "Nước đi": selected,
            "Hợp lệ": True
        })

uct_test_df = pd.DataFrame(uct_test_rows)
display(uct_test_df)

# Save experiment tables.
import os

# Tạo thư mục result
RESULT_DIR = "result"
os.makedirs(RESULT_DIR, exist_ok=True)

# Lưu kết quả Bài 6
match10.to_csv(
    os.path.join(RESULT_DIR, "bai6_ucb1_10_vs_random.csv"),
    index=False, encoding="utf-8-sig"
)

match100.to_csv(
    os.path.join(RESULT_DIR, "bai6_ucb1_100_vs_random.csv"),
    index=False, encoding="utf-8-sig"
)

comparison_df.to_csv(
    os.path.join(RESULT_DIR, "bai6_pmcs_vs_ucb1.csv"),
    index=False, encoding="utf-8-sig"
)

stability_df.to_csv(
    os.path.join(RESULT_DIR, "bai6_ucb1_stability.csv"),
    index=False, encoding="utf-8-sig"
)

uct_test_df.to_csv(
    os.path.join(RESULT_DIR, "bai6_uct_test_states.csv"),
    index=False, encoding="utf-8-sig"
)

print("Đã lưu toàn bộ kết quả Bài 6 vào:", os.path.abspath(RESULT_DIR))
assert uct_test_df["Hợp lệ"].all()
print("Bài 6: kiểm thử UCT và xuất bảng kết quả thành công.")


,Trạng thái,Ngân sách mô phỏng,Nước đi,Hợp lệ
0,X sắp thắng,10,2,True
1,X sắp thắng,100,8,True
2,O sắp thắng,10,2,True
3,O sắp thắng,100,6,True
4,X có thể hòa,10,3,True
5,X có thể hòa,100,7,True
6,Bàn trống,10,0,True
7,Bàn trống,100,6,True
8,Tình huống bất lợi,10,4,True
9,Tình huống bất lợi,100,4,True


Đã lưu toàn bộ kết quả Bài 6 vào: f:\TTNTNC\LAB05_GAME\result
Bài 6: kiểm thử UCT và xuất bảng kết quả thành công.


### Giải thích `UCT_depth1()` và UCB1

Ở độ sâu 1, mỗi hành động hợp lệ trên bàn cờ là một nhánh con của nút gốc. Thuật toán chọn nhánh bằng:
$$
UCB1(a) = \frac{Q(a)}{n(a)} + C \sqrt{\frac{\ln N}{n(a)}}
$$

- \(Q(a)/n(a)\): phần thưởng trung bình của hành động, đại diện cho **exploitation** (khai thác hành động đang có kết quả tốt).
- \(C\sqrt{\ln N/n(a)}\): bonus khám phá, đại diện cho **exploration** (ưu tiên hành động ít được mô phỏng).
- \(N\): tổng số lần mô phỏng ở nút cha; \(n(a)\): số lần mô phỏng hành động \(a\); \(C=\sqrt{2}\) điều khiển mức khám phá.

Mỗi hành động được thử ít nhất một lần khi ngân sách đủ lớn. Sau đó UCB1 phân bổ thêm mô phỏng cho các nhánh có trung bình phần thưởng tốt hoặc còn ít lượt thăm. Khi kết thúc, chọn hành động có số lượt thăm cao nhất.

In [52]:
# ĐÁNH GIÁ CHUẨN HÓA UCB1: N=10, 100, 1000; mỗi mức đấu Random 200 ván.
# Chỉ tính thời gian suy nghĩ và số nước đi của UCB1 agent.
import time, random, os
import pandas as pd
from collections import Counter

def make_ucb_agent(budget):
    def agent(board, player="x"):
        return UCT_depth1(board, N=budget, player=player)
    agent.__name__ = f"UCB1_N{budget}"
    return agent

def run_standard_ucb_tournament(agent, agent_name, games_per_side=100, seed=20261112):
    rows = []
    total_time = 0.0
    total_moves = 0
    total_wins = total_draws = total_losses = 0

    for agent_first in (True, False):
        counts = Counter()
        side_time = 0.0
        side_moves = 0
        for game_idx in range(games_per_side):
            np.random.seed(seed + (0 if agent_first else 100000) + game_idx)
            random.seed(seed + (0 if agent_first else 100000) + game_idx)
            board = empty_board()
            current = "x"
            while not terminal(board):
                is_agent_turn = ((current == "x") == agent_first)
                if is_agent_turn:
                    t0 = time.perf_counter()
                    move = agent(board, current)
                    side_time += time.perf_counter() - t0
                    side_moves += 1
                else:
                    move = random_player(board, current)
                board = result(board, current, move)
                current = other(current)

            outcome = check_board(board)
            agent_symbol = "x" if agent_first else "o"
            if outcome == "d": counts["draw"] += 1
            elif outcome == agent_symbol: counts["win"] += 1
            else: counts["loss"] += 1

        total_time += side_time
        total_moves += side_moves
        total_wins += counts["win"]
        total_draws += counts["draw"]
        total_losses += counts["loss"]
        rows.append({
            "Agent": agent_name,
            "Vị trí": "X (đi trước)" if agent_first else "O (đi sau)",
            "Thắng": counts["win"], "Hòa": counts["draw"], "Thua": counts["loss"],
            "Số ván": games_per_side,
            "Tổng thời gian agent (ms)": round(side_time * 1000, 4),
            "Số nước agent": side_moves,
            "Thời gian TB/nước (ms)": round(side_time * 1000 / side_moves, 6) if side_moves else 0
        })

    summary = {
        "Agent": agent_name, "Thắng": total_wins, "Hòa": total_draws, "Thua": total_losses,
        "Tổng số ván": 2 * games_per_side,
        "Tổng thời gian agent (ms)": round(total_time * 1000, 4),
        "Tổng số nước agent": total_moves,
        "Thời gian TB/nước (ms)": round(total_time * 1000 / total_moves, 6) if total_moves else 0
    }
    return pd.DataFrame(rows), summary

DEBUG = 0
ucb_standard_details = []
ucb_standard_summaries = []
for idx, budget in enumerate((10, 100, 1000)):
    agent_fn = make_ucb_agent(budget)
    detail, summary = run_standard_ucb_tournament(
        agent_fn, f"UCB1 N={budget}", seed=20261112 + idx * 10000
    )
    ucb_standard_details.append(detail)
    ucb_standard_summaries.append(summary)

ucb_standard_detail_df = pd.concat(ucb_standard_details, ignore_index=True)
ucb_standard_summary_df = pd.DataFrame(ucb_standard_summaries)
display(ucb_standard_detail_df)
display(ucb_standard_summary_df)

os.makedirs("result", exist_ok=True)
ucb_standard_detail_df.to_csv("result/bai6_ucb1_standard_detail.csv", index=False, encoding="utf-8-sig")
ucb_standard_summary_df.to_csv("result/bai6_ucb1_standard_summary.csv", index=False, encoding="utf-8-sig")


# PMCS (chia đều rollout) cũng được đo ở N=10, 100, 1000 theo cùng chuẩn 200 ván.
def make_pmcs_agent(budget):
    call_seed = {"value": 0}
    def agent(board, player="x"):
        legal = actions(board)
        if not legal:
            return None
        call_seed["value"] += 1
        rng = random.Random(20261112 + call_seed["value"])
        values = []
        base, rem = divmod(budget, len(legal))
        for idx, action in enumerate(legal):
            n_rollouts = base + (1 if idx < rem else 0)
            rewards = [playout(board, action, player=player) for _ in range(n_rollouts)]
            values.append(float(np.mean(rewards)) if rewards else -math.inf)
        return legal[int(np.argmax(values))]
    agent.__name__ = f"PMCS_N{budget}"
    return agent

pmcs_standard_details = []
pmcs_standard_summaries = []
for idx, budget in enumerate((10, 100, 1000)):
    agent_fn = make_pmcs_agent(budget)
    detail, summary = run_standard_ucb_tournament(
        agent_fn, f"PMCS N={budget}", seed=20261112 + 50000 + idx * 10000
    )
    pmcs_standard_details.append(detail)
    pmcs_standard_summaries.append(summary)

pmcs_standard_detail_df = pd.concat(pmcs_standard_details, ignore_index=True)
pmcs_standard_summary_df = pd.DataFrame(pmcs_standard_summaries)
display(pmcs_standard_detail_df)
display(pmcs_standard_summary_df)

pmcs_standard_detail_df.to_csv("result/bai6_pmcs_standard_detail.csv", index=False, encoding="utf-8-sig")
pmcs_standard_summary_df.to_csv("result/bai6_pmcs_standard_summary.csv", index=False, encoding="utf-8-sig")

# Bảng tổng hợp chuẩn để đối chiếu giữa nhóm: lấy N=100 làm mốc.
standard_group_comparison_df = pd.concat([
    ucb_standard_summary_df[ucb_standard_summary_df["Agent"] == "UCB1 N=100"],
    pmcs_standard_summary_df[pmcs_standard_summary_df["Agent"] == "PMCS N=100"]
], ignore_index=True)
display(standard_group_comparison_df)
standard_group_comparison_df.to_csv(
    "result/bai6_standard_group_comparison_N100.csv",
    index=False, encoding="utf-8-sig"
)


,Agent,Vị trí,Thắng,Hòa,Thua,Số ván,Tổng thời gian agent (ms),Số nước agent,Thời gian TB/nước (ms)
0,UCB1 N=10,X (đi trước),96,1,3,100,52.1332,359,0.145218
1,UCB1 N=10,O (đi sau),60,17,23,100,39.2077,324,0.121011
2,UCB1 N=100,X (đi trước),97,3,0,100,480.1777,332,1.446318
3,UCB1 N=100,O (đi sau),84,8,8,100,402.8441,321,1.254966
4,UCB1 N=1000,X (đi trước),100,0,0,100,4570.9090,324,14.107744
5,UCB1 N=1000,O (đi sau),89,6,5,100,3987.8326,325,12.270254


,Agent,Thắng,Hòa,Thua,Tổng số ván,Tổng thời gian agent (ms),Tổng số nước agent,Thời gian TB/nước (ms)
0,UCB1 N=10,156,18,26,200,91.3409,683,0.133735
1,UCB1 N=100,181,11,8,200,883.0218,653,1.352254
2,UCB1 N=1000,189,6,5,200,8558.7416,649,13.187583


,Agent,Vị trí,Thắng,Hòa,Thua,Số ván,Tổng thời gian agent (ms),Số nước agent,Thời gian TB/nước (ms)
0,PMCS N=10,X (đi trước),94,3,3,100,65.3389,368,0.177551
1,PMCS N=10,O (đi sau),70,5,25,100,45.0631,318,0.141708
2,PMCS N=100,X (đi trước),97,3,0,100,415.3512,339,1.225225
3,PMCS N=100,O (đi sau),84,10,6,100,339.9359,326,1.042748
4,PMCS N=1000,X (đi trước),99,1,0,100,3973.6357,327,12.151791
5,PMCS N=1000,O (đi sau),89,7,4,100,3257.3231,315,10.340708


,Agent,Thắng,Hòa,Thua,Tổng số ván,Tổng thời gian agent (ms),Tổng số nước agent,Thời gian TB/nước (ms)
0,PMCS N=10,164,8,28,200,110.4020,686,0.160936
1,PMCS N=100,181,13,6,200,755.2871,665,1.135770
2,PMCS N=1000,188,8,4,200,7230.9588,642,11.263176


,Agent,Thắng,Hòa,Thua,Tổng số ván,Tổng thời gian agent (ms),Tổng số nước agent,Thời gian TB/nước (ms)
0,UCB1 N=100,181,11,8,200,883.0218,653,1.352254
1,PMCS N=100,181,13,6,200,755.2871,665,1.135770
